# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


## Setup — Run this cell first


In [8]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('/content/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())


Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [9]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [19]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))

print(region_avg)

# Step 2: Build the chart
fig = px.bar(
    region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',
    text='Happiness_Score',
    title='Western Europe has the highest average happiness, while South Asia has the lowest',
    labels={
        'Happiness_Score': 'Average Happiness Score',
        'Region': ''
    }
)

fig.update_traces(
    marker_color='steelblue',
    texttemplate='%{x:.2f}',
    textposition='outside',
    cliponaxis=False
)

# Zero baseline
fig.update_xaxes(
    range=[0, region_avg['Happiness_Score'].max() * 1.15],
    title='Average Happiness Score'
)

fig.update_layout(
    template='simple_white',
    height=550,
    title_font_size=20,
    margin=dict(l=20, r=80, t=90, b=40)
)

fig.show()

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


### Task 1 — Insight

Western Europe has the highest average happiness score, at around 7.09. South Asia is much lower, with an average of about 3.62. So, there is quite a big difference in happiness between the regions in this dataset.

---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [20]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'

bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')

global_avg = df['Happiness_Score'].mean()

print(f"Global average: {global_avg:.2f}")

# Calculate the gap
happiness_gap = top8['Happiness_Score'].mean() - bottom8['Happiness_Score'].mean()
print(f"Average gap between Top 8 and Bottom 8: {happiness_gap:.2f}")

# Step 2: Build the chart
fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',
    color='Group',
    text='Happiness_Score',
    color_discrete_map={
        'Top 8': 'seagreen',
        'Bottom 8': 'indianred'
    },
    title='The happiest countries score far above the least happy countries',
    labels={
        'Happiness_Score': 'Happiness Score',
        'Country': '',
        'Group': 'Country Group'
    }
)

fig.update_traces(
    texttemplate='%{x:.2f}',
    textposition='outside',
    cliponaxis=False
)

# Zero baseline
fig.update_xaxes(
    range=[0, combined['Happiness_Score'].max() * 1.15],
    title='Happiness Score'
)

# Global average reference line
fig.add_vline(
    x=global_avg,
    line_dash='dash',
    line_width=2,
    annotation_text=f'Global average: {global_avg:.2f}',
    annotation_position='top'
)

# Visual separator between Bottom 8 and Top 8
fig.add_hline(
    y=7.5,
    line_dash='dot',
    line_width=2
)

# Labels for the two groups
fig.add_annotation(
    x=combined['Happiness_Score'].max() * 0.95,
    y=3.5,
    text='Bottom 8',
    showarrow=False,
    font=dict(size=14)
)

fig.add_annotation(
    x=combined['Happiness_Score'].max() * 0.95,
    y=11.5,
    text='Top 8',
    showarrow=False,
    font=dict(size=14)
)

fig.update_layout(
    template='simple_white',
    height=700,
    title_font_size=20,
    margin=dict(l=20, r=100, t=100, b=40)
)

fig.show()


Global average: 5.81
Average gap between Top 8 and Bottom 8: 4.16


### Task 2 — Insight

There is a clear difference between the top 8 and bottom 8 countries. The top group is mostly above the global average of 5.81, while the bottom group is below it. The gap between the two groups is quite noticeable.

In [21]:
# Stretch goal — grouped bar chart
# --------------------------------

# Select the five regions specified in the exercise
selected_regions = [
    'Western Europe',
    'Latin America and Caribbean',
    'East Asia',
    'Sub-Saharan Africa',
    'South Asia'
]

# Filter the dataset
stretch_df = df[df['Region'].isin(selected_regions)].copy()

# Calculate average GDP and Freedom for each region
regional_factors = (
    stretch_df.groupby('Region')[['GDP', 'Freedom']]
    .mean()
    .reset_index()
)

print(regional_factors)

# Convert to long format for grouped bars
regional_factors_long = regional_factors.melt(
    id_vars='Region',
    value_vars=['GDP', 'Freedom'],
    var_name='Factor',
    value_name='Average Value'
)

# Create grouped bar chart
fig = px.bar(
    regional_factors_long,
    x='Region',
    y='Average Value',
    color='Factor',
    barmode='group',
    text='Average Value',
    color_discrete_map={
        'GDP': 'steelblue',
        'Freedom': 'orange'
    },
    title='Western Europe has the highest average GDP among the selected regions',
    labels={
        'Region': '',
        'Average Value': 'Average Value',
        'Factor': 'Factor'
    }
)

fig.update_traces(
    texttemplate='%{y:.2f}',
    textposition='outside',
    cliponaxis=False
)

# Zero baseline
fig.update_yaxes(
    range=[0, regional_factors_long['Average Value'].max() * 1.15],
    title='Average Value'
)

fig.update_layout(
    template='simple_white',
    height=600,
    title_font_size=20,
    margin=dict(l=20, r=40, t=90, b=110)
)

fig.show()

                        Region        GDP   Freedom
0                    East Asia  10.312000  0.754000
1  Latin America and Caribbean   9.134923  0.754231
2                   South Asia   7.152250  0.546250
3           Sub-Saharan Africa   7.082000  0.599429
4               Western Europe  10.963533  0.896067


### Task 3 — Insight

Looking at these five regions, Western Europe has the highest average GDP. South Asia is lower for both GDP and Freedom compared with most of the other regions. Overall, the chart shows that the regions have quite different values for these two factors.